# 23. 구조 청킹 GTE/BGE reranker 비교 — CPU preflight

이 노트북은 Codex coder agent가 작성·실행한 개발셋 전용 실험 준비 기록이다. 현재 단계는 GPU0의 ollama 점유 때문에 모델 가중치 load와 scoring을 수행하지 않는다.


## 결과 전 고정 계약

- 검색 설정: 구조 청킹 k1=1.5, b=0.75, Vector:BM25=0.4:0.6, RRF k=60, Top20/Top50.
- 비교 시스템: no-reranker, all-GTE, all-BGE, selective-GTE, selective-BGE.
- selective 경로는 질문 문구만 분류해 proper/numeric은 RRF 유지, semantic만 reranker를 적용한다.
- 두 모델은 동일한 안전 제목 보강 입력만 사용한다. 점수 혼합 없이 raw single logit만으로 후보 내부를 재정렬하고 동점은 기존 RRF rank, chunk ID 순이다.
- 평가: Card10, Evidence20 전체 및 Numeric10/Semantic10의 Hit@3, Recall@5, MRR@5, nDCG@5, Card Hit/MRR.
- cross-corpus Recall/nDCG는 진단 전용이다. 개발 30질의 결과는 운영 승격이나 holdout 통과 근거가 아니다.
- 실행 금지: network/API/download/new embedding/package install/Chroma query. GPU scoring은 점유 해제 후 별도 승인 단계다.


In [1]:
from pathlib import Path
import csv, hashlib, json, os, re, statistics, unicodedata
import numpy as np
import pandas as pd

ROOT = Path.cwd() if Path.cwd().name == 'PickCardU' else Path.cwd().parent if Path.cwd().name == 'notebooks' and Path.cwd().parent.name == 'PickCardU' else (_ for _ in ()).throw(RuntimeError('cwd must be repo root or its notebooks directory'))
assert ROOT.name == 'PickCardU'
OUT = ROOT / 'notebooks/data/23_structural_chunking_reranker_comparison'
S13 = ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
S21 = ROOT / 'notebooks/data/21_current_chunking_prebranch_reranker_evaluation'
S22 = ROOT / 'notebooks/data/22_structural_heading_chunking_ablation'
GTE = ROOT / '.cache/reranker/gte-multilingual-reranker-base'
BGE = ROOT / '.cache/reranker/bge-reranker-v2-m3'
CUSTOM = ROOT / '.cache/huggingface/modules/transformers_modules/Alibaba_hyphen_NLP/new_hyphen_impl/40ced75c3017eb27626c9d4ea981bde21a2662f4'
CUSTOM_HUB = ROOT / '.cache/huggingface/hub/models--Alibaba-NLP--new-impl/snapshots/40ced75c3017eb27626c9d4ea981bde21a2662f4'
os.environ['HF_HOME'] = str((ROOT / '.cache/huggingface').resolve())
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'; os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
OUT.mkdir(parents=True, exist_ok=True)

GTE_REV = '8215cf04918ba6f7b6a62bb44238ce2953d8831c'
BGE_REV = '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
CUSTOM_REV = '40ced75c3017eb27626c9d4ea981bde21a2662f4'
RULE_TEXT = 'normalization=NFKC/lower/whitespace/terminal punctuation; precedence=proper>numeric>semantic; proper=identity intent for issuer/company/bank/card product; numeric_direct=얼마|몇 unit|얼마나 discount/accrual; numeric_target=end targets discount/accrual rate, annual fee, fee, amount/limit, monthly/annual limit, per-liter discount, mileage/point accrual criterion, spend amount/criterion, use count/period; standalone 이용료 excluded; 연회비 면제 조건 semantic; raw-digit-only disabled'
RULE_SHA = 'e4e2c0b33dc228382e269a3edb399b1b5b4d0c4f8d41149e96d4d06d64d55909'
CONFIG = 'k1_1.5_b_0.75_vector_0.4_bm25_0.6'

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def hash_files(paths):
    return {str(p.relative_to(ROOT)): sha256_file(p) for p in paths}

def tree_state(root):
    files = sorted(p for p in root.rglob('*') if p.is_file())
    mapping = {str(p.relative_to(root)): sha256_file(p) for p in files}
    digest = hashlib.sha256(''.join(k + ':' + mapping[k] + '\n' for k in sorted(mapping)).encode()).hexdigest()
    return {'file_count': len(files), 'total_bytes': sum(p.stat().st_size for p in files), 'tree_sha256': digest, 'files': mapping}

def load_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def normalize_query(value):
    text = unicodedata.normalize('NFKC', str(value)).lower()
    text = ' '.join(text.split())
    return re.sub(r'[?!。？！.]+$', '', text).strip()

PROPER = [
    ('proper_issuer_product', re.compile(r'(?:어느|어떤)\s*(?:(?:카드사|은행|회사)\s*)?상품(?:인가)?$')),
    ('proper_issuer_direct', re.compile(r'(?:어느\s*)?(?:카드사|은행|회사)(?:인가)?$')),
    ('proper_issuer_noun', re.compile(r'(?:발급사|발행사)(?:는|은|가|인가)?$')),
    ('proper_where_action', re.compile(r'어디서\s*(?:발급|발행|출시)')),
]
NUMERIC = [
    ('numeric_direct_amount', re.compile(r'얼마(?:인가|나)?$')),
    ('numeric_direct_count', re.compile(r'몇\s*(?:원|%|퍼센트|마일|마일리지|포인트|회|개월|일|년)(?:인가)?$')),
    ('numeric_direct_how_much', re.compile(r'얼마나\s*(?:할인|적립|차감|청구)')),
    ('numeric_target_end', re.compile(r'(?:할인율|적립률|연회비|수수료|(?:할인|적립)\s*(?:금액|한도)|(?:월|연간)\s*(?:할인|적립)?\s*한도|리터당\s*할인\s*금액|(?:마일리지|포인트)\s*적립\s*기준|실적\s*(?:금액|기준)|이용\s*(?:횟수|기간))(?:은|는|이|가|인가)?$')),
]

def classify_query(value):
    text = normalize_query(value)
    for rule_id, pattern in PROPER:
        match = pattern.search(text)
        if match:
            return 'proper_noun', rule_id, match.group(0), text
    if '연회비 면제 조건' not in text:
        for rule_id, pattern in NUMERIC:
            match = pattern.search(text)
            if match:
                return 'numeric_condition', rule_id, match.group(0), text
    return 'semantic', 'semantic_fallback', '', text

def augmented_text(chunk):
    metadata = chunk['metadata']
    path = [metadata['issuer'], metadata['card_name']] + list(chunk['heading_path'])
    return '[문서 경로]\n' + ' > '.join(path) + '\n\n[본문]\n' + chunk['body']

contract = {
    'schema_version': 'structural_reranker_preflight_v1',
    'provenance': 'Codex coder agent',
    'search': {'configuration': CONFIG, 'k1': 1.5, 'b': 0.75, 'vector_weight': 0.4, 'bm25_weight': 0.6, 'rrf_k': 60, 'depths': [20, 50]},
    'systems': ['no_reranker', 'all_gte', 'all_bge', 'selective_gte', 'selective_bge'],
    'classifier': {'rule_text': RULE_TEXT, 'rule_sha256': RULE_SHA, 'runtime_fields': ['query_text'], 'route': 'proper/numeric passthrough; semantic reranker'},
    'input': {'template': '[문서 경로] issuer > card_name > heading_path; [본문] body', 'allowlist': ['metadata.issuer', 'metadata.card_name', 'heading_path', 'body']},
    'ranking': {'score': 'raw single logit descending', 'tie_break': ['original_rrf_rank', 'chunk_id'], 'rrf_logit_mixing': False, 'top20_exact_prefix_of_top50': True},
    'models': {
        'gte': {'path': str(GTE.relative_to(ROOT)), 'revision': GTE_REV, 'custom_code_revision': CUSTOM_REV, 'trust_remote_code': True, 'explicit_opt_in_required': True},
        'bge': {'path': str(BGE.relative_to(ROOT)), 'revision': BGE_REV, 'architecture': 'XLMRobertaForSequenceClassification', 'trust_remote_code': False},
    },
    'inference': {'max_length': 8192, 'truncation': 'only_second', 'padding': 'dynamic', 'dtype': 'float16', 'batch_size': 2, 'physical_gpu': 0},
    'evaluation': {'groups': ['card10', 'evidence20', 'numeric10', 'semantic10'], 'metrics': ['card_hit_at_3', 'card_mrr_at_5', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5'], 'cross_corpus_recall_ndcg': 'diagnostic_only'},
    'selection': {'all_query_first': True, 'all_query_requires_nonregression_all_numeric_semantic': True, 'fallback': 'selective route only', 'promotion_eligible': False},
    'execution': {'environment': 'skn25', 'network': 0, 'api': 0, 'downloads': 0, 'new_embeddings': 0, 'package_installs': 0, 'chroma_queries': 0, 'current_stage': 'shared GPU0 execution explicitly approved', 'resource_measurement_status': 'shared_gpu_measurement_non_confirmatory', 'oom_policy': 'batch2; on OOM discard partial and restart both models at batch1 only; batch1 failure is infeasible'},
}
(OUT / 'evaluation_contract.json').write_text(json.dumps(contract, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


3197

In [2]:
source_paths = [
    S13 / 'retrieval_per_query.csv',
    S21 / 'phase_a_classification.csv', S21 / 'evaluation_contract.json',
    S21 / 'leaf_per_query_metrics.csv', S21 / 'leaf_selective_routes.csv', S21 / 'resources.json',
    S22 / 'chunks.jsonl', S22 / 'relevance_sets.jsonl', S22 / 'followup2_rankings.csv',
    S22 / 'followup2_contract.json', S22 / 'followup2_per_query.csv',
    S22 / 'followup1_summary.json', S22 / 'followup1_per_query.csv',
]
assert all(path.is_file() for path in source_paths)
source_before = hash_files(source_paths)
gte_before, bge_before, custom_before, custom_hub_before = tree_state(GTE), tree_state(BGE), tree_state(CUSTOM), tree_state(CUSTOM_HUB)

gte_meta = (GTE / '.cache/huggingface/download/config.json.metadata').read_text().splitlines()[0]
bge_meta = (BGE / '.cache/huggingface/download/config.json.metadata').read_text().splitlines()[0]
gte_config = json.loads((GTE / 'config.json').read_text())
bge_config = json.loads((BGE / 'config.json').read_text())
assert gte_meta == GTE_REV and bge_meta == BGE_REV and CUSTOM.name == CUSTOM_REV and CUSTOM_HUB.name == CUSTOM_REV
assert gte_config['auto_map']['AutoModelForSequenceClassification'].endswith('NewForSequenceClassification')
assert bge_config['architectures'] == ['XLMRobertaForSequenceClassification']
assert bge_config['max_position_embeddings'] == 8194

chunks = load_jsonl(S22 / 'chunks.jsonl')
chunks_by_id = {row['chunk_id']: row for row in chunks}
assert len(chunks) == len(chunks_by_id) == 147
rankings_all = pd.read_csv(S22 / 'followup2_rankings.csv')
rankings = rankings_all[rankings_all.configuration == CONFIG].copy()
assert len(rankings) == 30 and rankings.query_id.nunique() == 30
for column in ['vector_top50_chunk_ids', 'bm25_top50_chunk_ids', 'fused_top50_chunk_ids', 'fused_top50_scores']:
    rankings[column] = rankings[column].map(json.loads)
assert all(len(ids) == len(set(ids)) == 50 for ids in rankings.fused_top50_chunk_ids)
assert all(set(ids) <= set(chunks_by_id) for ids in rankings.fused_top50_chunk_ids)
top20_prefix_count = sum(ids[:20] == list(ids)[:20] for ids in rankings.fused_top50_chunk_ids)
assert top20_prefix_count == 30

queries_all = pd.read_csv(S13 / 'retrieval_per_query.csv')
queries = queries_all[queries_all.method == 'keyword'].copy()
assert len(queries) == queries.query_id.nunique() == 30
saved_class = pd.read_csv(S21 / 'phase_a_classification.csv')
class_rows = []
for row in queries.itertuples(index=False):
    predicted, rule_id, span, normalized = classify_query(row.query)
    class_rows.append({'query_id': row.query_id, 'gold_category': row.category, 'predicted_category': predicted, 'matched_rule_id': rule_id, 'matched_span': span, 'normalized_query': normalized, 'correct': predicted == row.category})
classification = pd.DataFrame(class_rows).sort_values('query_id').reset_index(drop=True)
saved = saved_class[['query_id', 'gold_category', 'predicted_category', 'matched_rule_id', 'matched_span', 'normalized_query', 'correct']].sort_values('query_id').reset_index(drop=True)
classification['matched_span'] = classification['matched_span'].fillna('')
saved['matched_span'] = saved['matched_span'].fillna('')
assert (classification.matched_span == '').sum() == (saved.matched_span == '').sum() == 10
pd.testing.assert_frame_equal(classification, saved, check_dtype=False)
assert classification.correct.all() and classification.predicted_category.value_counts().to_dict() == {'numeric_condition': 10, 'semantic': 10, 'proper_noun': 10}
classification.to_csv(OUT / 'preflight_classification.csv', index=False)

query_map = queries.set_index('query_id').to_dict('index')
pair_rows = []
for row in rankings.itertuples(index=False):
    query = query_map[row.query_id]['query']
    for rank, chunk_id in enumerate(row.fused_top50_chunk_ids, 1):
        chunk = chunks_by_id[chunk_id]
        text = augmented_text(chunk)
        pair_rows.append({'query_id': row.query_id, 'chunk_id': chunk_id, 'rrf_rank': rank, 'query_sha256': hashlib.sha256(query.encode()).hexdigest(), 'augmented_text_sha256': hashlib.sha256(text.encode()).hexdigest(), 'issuer': chunk['metadata']['issuer'], 'card_name': chunk['metadata']['card_name'], 'heading_count': len(chunk['heading_path']), 'body_chars': len(chunk['body']), 'input_chars': len(text), 'gold_fields_used': False})
pairs = pd.DataFrame(pair_rows)
assert len(pairs) == 1500 and not pairs.duplicated(['query_id', 'chunk_id']).any()
assert pairs.groupby('query_id').size().eq(50).all() and not pairs.gold_fields_used.any()
pairs.to_csv(OUT / 'augmented_input_audit.csv', index=False)

relevance_rows = [row for row in load_jsonl(S22 / 'relevance_sets.jsonl') if row['configuration'] == 'structural_heading_path']
assert len(relevance_rows) == 20
relevance = {row['query_id']: set(row['relevant_chunk_ids']) for row in relevance_rows}
ceiling_rows = []
for row in rankings.itertuples(index=False):
    q = query_map[row.query_id]
    for depth in (20, 50):
        ids = row.fused_top50_chunk_ids[:depth]
        expected_card = q['expected_card']
        card_hits = [cid for cid in ids if chunks_by_id[cid]['metadata']['card_key'] == expected_card]
        relevant = relevance.get(row.query_id, set())
        strict_hits = [cid for cid in ids if cid in relevant]
        ceiling_rows.append({'query_id': row.query_id, 'category': q['category'], 'depth': depth, 'candidate_count': len(ids), 'expected_card_candidate_hit': int(bool(card_hits)), 'expected_card_candidate_count': len(card_hits), 'answer_candidate_hit': int(bool(strict_hits)) if relevant else np.nan, 'answer_candidate_recall': len(strict_hits) / len(relevant) if relevant else np.nan, 'missing_relevant_ids': json.dumps(sorted(relevant - set(ids)), ensure_ascii=False)})
ceilings = pd.DataFrame(ceiling_rows)
assert len(ceilings) == 60 and set(ceilings.candidate_count) == {20, 50}
ceilings.to_csv(OUT / 'candidate_ceiling.csv', index=False)

from transformers import AutoTokenizer
token_stats = {}
for model_name, model_path in [('gte', GTE), ('bge', BGE)]:
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, trust_remote_code=False)
    lengths, truncated = [], 0
    for ranking_row in rankings.itertuples(index=False):
        query = query_map[ranking_row.query_id]['query']
        for chunk_id in ranking_row.fused_top50_chunk_ids:
            document = augmented_text(chunks_by_id[chunk_id])
            full_len = len(tokenizer(query, document, add_special_tokens=True, truncation=False)['input_ids'])
            used_len = len(tokenizer(query, document, add_special_tokens=True, truncation='only_second', max_length=8192)['input_ids'])
            assert used_len <= 8192
            lengths.append(used_len)
            truncated += int(full_len > used_len)
    token_stats[model_name] = {'pairs': len(lengths), 'input_tokens_total': int(sum(lengths)), 'p50': float(np.percentile(lengths, 50)), 'p95': float(np.percentile(lengths, 95)), 'max': int(max(lengths)), 'truncated_pairs': truncated, 'model_max_length': int(tokenizer.model_max_length)}
assert token_stats['gte']['pairs'] == token_stats['bge']['pairs'] == 1500
assert token_stats['bge']['model_max_length'] == 8192

source_after = hash_files(source_paths)
gte_after, bge_after, custom_after, custom_hub_after = tree_state(GTE), tree_state(BGE), tree_state(CUSTOM), tree_state(CUSTOM_HUB)
assert source_after == source_before
assert gte_after == gte_before and bge_after == bge_before and custom_after == custom_before and custom_hub_after == custom_hub_before
preflight = {'status': 'passed_before_shared_gpu_scoring', 'query_count': 30, 'category_counts': classification.predicted_category.value_counts().to_dict(), 'chunk_count': 147, 'top20_pairs': 600, 'top50_pairs': 1500, 'unique_query_chunk_pairs': 1500, 'token_stats': token_stats, 'model_cache': {'gte': {k: gte_before[k] for k in ['file_count', 'total_bytes', 'tree_sha256']}, 'bge': {k: bge_before[k] for k in ['file_count', 'total_bytes', 'tree_sha256']}, 'custom_code': {k: custom_before[k] for k in ['file_count', 'total_bytes', 'tree_sha256']}, 'custom_hub': {k: custom_hub_before[k] for k in ['file_count', 'total_bytes', 'tree_sha256']}}, 'model_loads': 0, 'scored_pairs': 0, 'gpu_compute': 0, 'network_api_download_embedding_chroma': 0, 'shared_gpu_note': 'physical GPU0 has ollama using 256 MiB; shared execution explicitly approved'}
(OUT / 'preflight.json').write_text(json.dumps(preflight, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
integrity = {'status': 'passed_cpu_preflight', 'source_hashes_before': source_before, 'source_hashes_after': source_after, 'source_unchanged': True, 'model_cache_unchanged': True, 'query30': True, 'chunks147': True, 'top20_prefix_30_of_30': True, 'classification_exact_notebook21': True, 'no_leak_allowlist': True, 'finite_token_lengths': True, 'gpu_model_scoring_executed': False}
(OUT / 'integrity.json').write_text(json.dumps(integrity, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
readme = '''# 23 구조 청킹 reranker 비교 — CPU preflight

현재 저장 결과는 개발 질의 30개와 구조 청크 147개의 입력·후보·분류·로컬 모델 캐시를 확인한 사전 점검이다. GPU0에 ollama가 256 MiB를 사용 중이어서 모델 가중치 load와 scoring은 시작하지 않았다.

- Top20: 각 질의 Top50의 정확한 앞 20개, 총 600 pair
- Top50: 질의별 50개, 총 1,500 unique query-chunk pair
- 분류: 질문 문구만 사용하며 notebook21 저장 결과와 30/30 일치
- 모델 입력: issuer, card_name, heading_path, body만 사용. gold/relevance 필드는 사용하지 않음
- 외부 호출: network/API/download/new embedding/Chroma query 0

Hit@3는 상위 3개 안에 관련 결과가 하나라도 있는지, Recall@5는 전체 관련 결과 중 상위 5개가 회수한 비율, MRR@5는 첫 관련 결과가 얼마나 앞에 있는지, nDCG@5는 관련 결과가 앞쪽에 모였는지를 뜻한다. 실제 성능·속도·VRAM 결과는 GPU 점유 해제 후 별도 실행해야 한다. 이 실험은 개발셋 진단이며 운영 확정이나 holdout 검증이 아니다.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8')
print(json.dumps(preflight, ensure_ascii=False, indent=2))


/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "status": "passed_before_shared_gpu_scoring",
  "query_count": 30,
  "category_counts": {
    "proper_noun": 10,
    "numeric_condition": 10,
    "semantic": 10
  },
  "chunk_count": 147,
  "top20_pairs": 600,
  "top50_pairs": 1500,
  "unique_query_chunk_pairs": 1500,
  "token_stats": {
    "gte": {
      "pairs": 1500,
      "input_tokens_total": 556774,
      "p50": 280.0,
      "p95": 1185.05,
      "max": 1647,
      "truncated_pairs": 0,
      "model_max_length": 32768
    },
    "bge": {
      "pairs": 1500,
      "input_tokens_total": 556774,
      "p50": 280.0,
      "p95": 1185.05,
      "max": 1647,
      "truncated_pairs": 0,
      "model_max_length": 8192
    }
  },
  "model_cache": {
    "gte": {
      "file_count": 27,
      "total_bytes": 629269531,
      "tree_sha256": "b14ecc08279610a6f08ce1d82bbbd54feca4729fde2cae531e8fd435dba21bef"
    },
    "bge": {
      "file_count": 40,
      "total_bytes": 2293568873,
      "tree_sha256": "ddfd5c5b6a8c9c3c7421a3d423da60f4a8

## 승인된 shared GPU0 scoring

GPU0의 ollama 256 MiB 점유를 알고도 사용자가 공유 실행을 승인했다. ollama를 중단하거나 변경하지 않고 실행 전후 GPU0 process/util/memory snapshot을 저장한다. 자원 수치는 shared_gpu_measurement_non_confirmatory이며 품질 평가는 정상 수행한다.


In [3]:
assert os.environ.get('RUN_APPROVED_23_GPU') == '1'
assert os.environ.get('ALLOW_GTE_REMOTE_CODE') == '1'
assert os.environ.get('CUDA_VISIBLE_DEVICES') == '0'
import gc, subprocess, time

def gpu_snapshot():
    gpu_lines = subprocess.run(['nvidia-smi', '--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'], check=True, capture_output=True, text=True).stdout.strip().splitlines()
    gpu0 = [line for line in gpu_lines if line.split(',')[0].strip() == '0']
    assert len(gpu0) == 1
    gpu_uuid = gpu0[0].split(',')[1].strip()
    process_text = subprocess.run(['nvidia-smi', '--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory', '--format=csv,noheader,nounits'], check=True, capture_output=True, text=True).stdout.strip()
    processes = []
    for line in process_text.splitlines():
        parts = [part.strip() for part in line.split(',', 3)]
        if len(parts) == 4 and parts[0] == gpu_uuid:
            processes.append({'gpu_uuid': parts[0], 'pid': int(parts[1]), 'process_name': parts[2], 'used_memory_mib': float(parts[3])})
    return {'gpu_line': gpu0[0], 'processes': processes, 'captured_at_unix': time.time()}

gpu_before = gpu_snapshot()
assert any('ollama' in row['process_name'].lower() for row in gpu_before['processes'])
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert '3090' in torch.cuda.get_device_name(0)

pair_inputs = []
for ranking_row in rankings.itertuples(index=False):
    query = query_map[ranking_row.query_id]['query']
    for rrf_rank, chunk_id in enumerate(ranking_row.fused_top50_chunk_ids, 1):
        pair_inputs.append((ranking_row.query_id, chunk_id, rrf_rank, query, augmented_text(chunks_by_id[chunk_id])))
assert len(pair_inputs) == 1500 and len({(row[0], row[1]) for row in pair_inputs}) == 1500

score_rows, resource_rows = [], []
model_specs = [
    ('gte', GTE, True, {'code_revision': CUSTOM_REV}),
    ('bge', BGE, False, {}),
]
for model_name, model_path, trust_remote_code, extra in model_specs:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    load_started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True, trust_remote_code=trust_remote_code, dtype=torch.float16, **extra).eval().to('cuda:0')
    load_seconds = time.perf_counter() - load_started
    warmup_started = time.perf_counter()
    warm = tokenizer(['준비'], ['준비'], padding=True, truncation='only_second', max_length=8192, return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        warm_logits = model(**warm).logits.reshape(-1)
    assert torch.isfinite(warm_logits).all()
    torch.cuda.synchronize()
    warmup_seconds = time.perf_counter() - warmup_started
    del warm, warm_logits
    scoring_started = time.perf_counter()
    model_scores = []
    try:
        for start in range(0, len(pair_inputs), 2):
            batch = pair_inputs[start:start + 2]
            encoded = tokenizer([row[3] for row in batch], [row[4] for row in batch], padding=True, truncation='only_second', max_length=8192, return_tensors='pt').to('cuda:0')
            with torch.inference_mode():
                logits = model(**encoded).logits.reshape(-1).float().cpu().numpy()
            assert len(logits) == len(batch) and np.isfinite(logits).all()
            model_scores.extend(float(value) for value in logits)
            del encoded, logits
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('OOM at batch2: discard partial and restart full fresh kernel with both models at batch1 only') from exc
    torch.cuda.synchronize()
    scoring_seconds = time.perf_counter() - scoring_started
    assert len(model_scores) == 1500 and np.isfinite(np.asarray(model_scores)).all()
    for pair, score in zip(pair_inputs, model_scores):
        score_rows.append({'model': model_name, 'query_id': pair[0], 'chunk_id': pair[1], 'original_rrf_rank': pair[2], 'raw_logit': score})
    resource_rows.append({'model': model_name, 'load_seconds': load_seconds, 'warmup_seconds': warmup_seconds, 'scoring_seconds': scoring_seconds, 'quality_pairs': 1500, 'pairs_per_second': 1500 / scoring_seconds, 'batch_size': 2, 'dtype': 'float16', 'max_length': 8192, 'truncation': 'only_second', 'oom': False, 'peak_allocated_gib': torch.cuda.max_memory_allocated() / 2**30, 'peak_reserved_gib': torch.cuda.max_memory_reserved() / 2**30, 'measurement_status': 'shared_gpu_measurement_non_confirmatory', 'token_p50': token_stats[model_name]['p50'], 'token_p95': token_stats[model_name]['p95'], 'token_max': token_stats[model_name]['max'], 'truncated_pairs': token_stats[model_name]['truncated_pairs'], 'cache_bytes': gte_before['total_bytes'] if model_name == 'gte' else bge_before['total_bytes'], 'revision': GTE_REV if model_name == 'gte' else BGE_REV})
    del model, tokenizer, model_scores
    gc.collect()
    torch.cuda.empty_cache()

pair_scores = pd.DataFrame(score_rows)
assert len(pair_scores) == 3000 and not pair_scores.duplicated(['model', 'query_id', 'chunk_id']).any()
assert pair_scores.groupby(['model', 'query_id']).size().eq(50).all() and np.isfinite(pair_scores.raw_logit).all()
pair_scores.to_csv(OUT / 'pair_scores.csv', index=False)
gpu_after = gpu_snapshot()
resources = {'measurement_status': 'shared_gpu_measurement_non_confirmatory', 'physical_gpu': 0, 'cuda_visible_devices': '0', 'gpu_before': gpu_before, 'gpu_after': gpu_after, 'models': resource_rows, 'network_api_download_new_embedding_chroma': 0}
(OUT / 'resources.json').write_text(json.dumps(resources, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(resources, ensure_ascii=False, indent=2))


{
  "measurement_status": "shared_gpu_measurement_non_confirmatory",
  "physical_gpu": 0,
  "cuda_visible_devices": "0",
  "gpu_before": {
    "gpu_line": "0, GPU-7ba60a9d-7e53-b0e1-1b30-a1a0e6480ce5, NVIDIA GeForce RTX 3090, 279, 24576, 0",
    "processes": [
      {
        "gpu_uuid": "GPU-7ba60a9d-7e53-b0e1-1b30-a1a0e6480ce5",
        "pid": 347458,
        "process_name": "/usr/bin/ollama",
        "used_memory_mib": 256.0
      }
    ],
    "captured_at_unix": 1787577703.7965007
  },
  "gpu_after": {
    "gpu_line": "0, GPU-7ba60a9d-7e53-b0e1-1b30-a1a0e6480ce5, NVIDIA GeForce RTX 3090, 621, 24576, 76",
    "processes": [
      {
        "gpu_uuid": "GPU-7ba60a9d-7e53-b0e1-1b30-a1a0e6480ce5",
        "pid": 347458,
        "process_name": "/usr/bin/ollama",
        "used_memory_mib": 256.0
      },
      {
        "gpu_uuid": "GPU-7ba60a9d-7e53-b0e1-1b30-a1a0e6480ce5",
        "pid": 3932323,
        "process_name": "/home/sms/anaconda3/envs/skn25/bin/python",
        "used_memory

In [4]:
def reciprocal_rank(ids, relevant, limit=5):
    for rank, chunk_id in enumerate(ids[:limit], 1):
        if chunk_id in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_5(ids, relevant):
    dcg = sum((1.0 / np.log2(rank + 1)) for rank, chunk_id in enumerate(ids[:5], 1) if chunk_id in relevant)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return dcg / idcg if idcg else np.nan

score_map = {(row.model, row.query_id, row.chunk_id): row.raw_logit for row in pair_scores.itertuples(index=False)}
class_map = classification.set_index('query_id').predicted_category.to_dict()
systems = ['no_reranker', 'all_gte', 'all_bge', 'selective_gte', 'selective_bge']
per_query_rows = []
for ranking_row in rankings.itertuples(index=False):
    qid = ranking_row.query_id
    q = query_map[qid]
    for depth in (20, 50):
        base = ranking_row.fused_top50_chunk_ids[:depth]
        assert base == ranking_row.fused_top50_chunk_ids[:depth]
        model_rankings = {}
        for model_name in ('gte', 'bge'):
            model_rankings[model_name] = sorted(base, key=lambda cid: (-score_map[(model_name, qid, cid)], base.index(cid) + 1, cid))
            assert set(model_rankings[model_name]) == set(base) and len(model_rankings[model_name]) == depth
        rankings_by_system = {
            'no_reranker': base,
            'all_gte': model_rankings['gte'],
            'all_bge': model_rankings['bge'],
            'selective_gte': model_rankings['gte'] if class_map[qid] == 'semantic' else base,
            'selective_bge': model_rankings['bge'] if class_map[qid] == 'semantic' else base,
        }
        for system, ids in rankings_by_system.items():
            card_relevant = {cid for cid in chunks_by_id if chunks_by_id[cid]['metadata']['card_key'] == q['expected_card']}
            answer_relevant = relevance.get(qid, set())
            top5 = ids[:5]
            per_query_rows.append({'depth': depth, 'system': system, 'query_id': qid, 'question_group': 'card' if q['category'] == 'proper_noun' else 'evidence', 'category': q['category'], 'candidate_count': depth, 'route': 'reranker' if system.startswith('all_') and system != 'no_reranker' or system.startswith('selective_') and class_map[qid] == 'semantic' else 'no_reranker', 'card_hit_at_3': int(any(cid in card_relevant for cid in ids[:3])), 'card_mrr_at_5': reciprocal_rank(ids, card_relevant), 'hit_at_3': int(any(cid in answer_relevant for cid in ids[:3])) if answer_relevant else np.nan, 'recall_at_5': len(set(ids[:5]) & answer_relevant) / len(answer_relevant) if answer_relevant else np.nan, 'mrr_at_5': reciprocal_rank(ids, answer_relevant) if answer_relevant else np.nan, 'ndcg_at_5': ndcg_at_5(ids, answer_relevant), 'relevant_count': len(answer_relevant) if answer_relevant else np.nan, 'top5_chunk_ids': json.dumps(top5, ensure_ascii=False), 'top5_cards': json.dumps([chunks_by_id[cid]['metadata']['card_key'] for cid in top5], ensure_ascii=False)})
per_query = pd.DataFrame(per_query_rows)
assert len(per_query) == 300 and not per_query.duplicated(['depth', 'system', 'query_id']).any()
assert per_query.groupby(['depth', 'system']).size().eq(30).all()
assert per_query.groupby(['depth', 'system', 'query_id']).size().eq(1).all()
for metric in ['card_hit_at_3', 'card_mrr_at_5', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']:
    values = per_query[metric].dropna()
    assert values.between(0, 1).all()
per_query.to_csv(OUT / 'per_query_metrics.csv', index=False)

stored = pd.read_csv(S22 / 'followup2_per_query.csv')
stored = stored[stored.configuration == CONFIG].sort_values('query_id').reset_index(drop=True)
assert len(stored) == 30
for depth in (20, 50):
    current = per_query[(per_query.depth == depth) & (per_query.system == 'no_reranker')].sort_values('query_id').reset_index(drop=True)
    assert current.top5_chunk_ids.map(json.loads).tolist() == stored.top5_chunk_ids.map(json.loads).tolist()
    for left, right in [('card_hit_at_3', 'card_hit_at_3'), ('card_mrr_at_5', 'card_mrr_at_5'), ('hit_at_3', 'answer_hit_at_3'), ('recall_at_5', 'answer_recall_at_5'), ('mrr_at_5', 'answer_mrr_at_5'), ('ndcg_at_5', 'answer_ndcg_at_5')]:
        assert np.allclose(current[left].to_numpy(float), stored[right].to_numpy(float), rtol=0, atol=1e-12, equal_nan=True)

group_masks = {
    'all': lambda frame: pd.Series(True, index=frame.index),
    'card': lambda frame: frame.category == 'proper_noun',
    'evidence': lambda frame: frame.category != 'proper_noun',
    'numeric': lambda frame: frame.category == 'numeric_condition',
    'semantic': lambda frame: frame.category == 'semantic',
}
metric_columns = ['card_hit_at_3', 'card_mrr_at_5', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']
summary_rows = []
for (depth, system), frame in per_query.groupby(['depth', 'system'], sort=True):
    for group, mask_fn in group_masks.items():
        selected = frame[mask_fn(frame)]
        row = {'depth': depth, 'system': system, 'group': group, 'denominator': len(selected)}
        for metric in metric_columns:
            row[metric] = float(selected[metric].mean()) if selected[metric].notna().any() else np.nan
            row[metric + '_denominator'] = int(selected[metric].notna().sum())
        summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
assert len(summary) == 50
summary.to_csv(OUT / 'summary.csv', index=False)

delta_rows = []
for depth in (20, 50):
    baseline = per_query[(per_query.depth == depth) & (per_query.system == 'no_reranker')].set_index('query_id')
    for system in systems[1:]:
        candidate = per_query[(per_query.depth == depth) & (per_query.system == system)].set_index('query_id')
        for qid in baseline.index:
            for metric in metric_columns:
                left, right = baseline.at[qid, metric], candidate.at[qid, metric]
                if pd.notna(left) and pd.notna(right):
                    delta_rows.append({'depth': depth, 'comparison': system + '_vs_no_reranker', 'system': system, 'query_id': qid, 'category': baseline.at[qid, 'category'], 'metric': metric, 'baseline': left, 'candidate': right, 'delta': right - left})
paired = pd.DataFrame(delta_rows)
paired.to_csv(OUT / 'paired_deltas.csv', index=False)
wlt_rows = []
for (depth, comparison, metric), frame in paired.groupby(['depth', 'comparison', 'metric']):
    for group, categories in {'all': None, 'card': {'proper_noun'}, 'evidence': {'numeric_condition', 'semantic'}, 'numeric': {'numeric_condition'}, 'semantic': {'semantic'}}.items():
        selected = frame if categories is None else frame[frame.category.isin(categories)]
        if len(selected):
            wlt_rows.append({'depth': depth, 'comparison': comparison, 'group': group, 'metric': metric, 'denominator': len(selected), 'wins': int((selected.delta > 1e-12).sum()), 'losses': int((selected.delta < -1e-12).sum()), 'ties': int((selected.delta.abs() <= 1e-12).sum()), 'mean_delta': float(selected.delta.mean())})
wlt = pd.DataFrame(wlt_rows)
assert (wlt.wins + wlt.losses + wlt.ties == wlt.denominator).all()
wlt.to_csv(OUT / 'paired_wlt.csv', index=False)

old_leaf = pd.read_csv(S21 / 'leaf_per_query_metrics.csv')
old_routes = pd.read_csv(S21 / 'leaf_selective_routes.csv')
old_card = pd.read_csv(S22 / 'followup1_per_query.csv')
cross_rows = []
for depth in (20, 50):
    variant = 'leaf_only_top20' if depth == 20 else 'leaf_available_from_original_top50'
    for system, old_system in {'no_reranker': 'no_reranker', 'all_gte': 'gte_augmented', 'all_bge': 'bge_augmented'}.items():
        old = old_leaf[(old_leaf.weight == 'vector_0.4_bm25_0.6') & (old_leaf.variant == variant) & (old_leaf.system == old_system) & (old_leaf.view == 'answer_bearing_raw')]
        assert len(old) == 20
        current = per_query[(per_query.depth == depth) & (per_query.system == system) & (per_query.question_group == 'evidence')]
        merged = current.merge(old[['query_id', 'card_hit_at_3', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']], on='query_id', suffixes=('_new', '_old'), validate='one_to_one')
        for row in merged.itertuples(index=False):
            cross_rows.append({'depth': depth, 'system': system, 'query_id': row.query_id, 'category': row.category, 'old_candidate_scope': variant, 'new_candidate_count': row.candidate_count, 'old_candidate_count': int(old[old.query_id == row.query_id].candidate_count.iloc[0]), **{metric + '_delta': getattr(row, metric + '_new') - getattr(row, metric + '_old') for metric in ['card_hit_at_3', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']}})
    for system, reranker in {'selective_gte': 'gte_augmented', 'selective_bge': 'bge_augmented'}.items():
        old = old_routes[(old_routes.weight == 'vector_0.4_bm25_0.6') & (old_routes.variant == variant) & (old_routes.reranker == reranker) & (old_routes.route == 'actual_regex') & (old_routes.view == 'answer_bearing_raw')]
        assert len(old) == 20
        current = per_query[(per_query.depth == depth) & (per_query.system == system) & (per_query.question_group == 'evidence')]
        merged = current.merge(old[['query_id', 'card_hit_at_3', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']], on='query_id', suffixes=('_new', '_old'), validate='one_to_one')
        for row in merged.itertuples(index=False):
            old_count = int(old_leaf[(old_leaf.weight == 'vector_0.4_bm25_0.6') & (old_leaf.variant == variant) & (old_leaf.system == 'no_reranker') & (old_leaf.query_id == row.query_id) & (old_leaf.view == 'answer_bearing_raw')].candidate_count.iloc[0])
            cross_rows.append({'depth': depth, 'system': system, 'query_id': row.query_id, 'category': row.category, 'old_candidate_scope': variant, 'new_candidate_count': row.candidate_count, 'old_candidate_count': old_count, **{metric + '_delta': getattr(row, metric + '_new') - getattr(row, metric + '_old') for metric in ['card_hit_at_3', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']}})
cross = pd.DataFrame(cross_rows)
assert len(cross) == 200
cross.to_csv(OUT / 'cross_pipeline_comparison.csv', index=False)
cross_summary = cross.groupby(['depth', 'system', 'category'], as_index=False).agg({column: 'mean' for column in cross.columns if column.endswith('_delta')})
cross_summary.to_csv(OUT / 'cross_pipeline_summary.csv', index=False)

def summary_value(system, group, metric, depth=50):
    rows = summary[(summary.depth == depth) & (summary.system == system) & (summary.group == group)]
    assert len(rows) == 1
    return float(rows.iloc[0][metric])

guardrail_metrics = [('evidence', metric) for metric in ['hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']] + [('numeric', metric) for metric in ['hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']] + [('semantic', metric) for metric in ['hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']] + [('card', metric) for metric in ['card_hit_at_3', 'card_mrr_at_5']]
candidate_checks = {}
for system in systems[1:]:
    checks = {group + ':' + metric: summary_value(system, group, metric) + 1e-12 >= summary_value('no_reranker', group, metric) for group, metric in guardrail_metrics}
    candidate_checks[system] = {'vs_new_no_reranker': checks, 'pass_vs_new_no_reranker': all(checks.values())}

old_op_evidence = old_routes[(old_routes.weight == 'vector_0.4_bm25_0.6') & (old_routes.variant == 'leaf_available_from_original_top50') & (old_routes.reranker == 'bge_augmented') & (old_routes.route == 'actual_regex') & (old_routes.view == 'answer_bearing_raw')]
assert len(old_op_evidence) == 20
old_op_card = old_card[(old_card.configuration == 'existing_exact_v1_leaf_filtered') & (old_card.category == 'proper_noun')]
assert len(old_op_card) == 10
old_op = {'card': {'card_hit_at_3': float(old_op_card.card_hit_at_3.mean()), 'card_mrr_at_5': float(old_op_card.card_mrr_at_5.mean())}}
for group, categories in {'evidence': {'numeric_condition', 'semantic'}, 'numeric': {'numeric_condition'}, 'semantic': {'semantic'}}.items():
    rows = old_op_evidence[old_op_evidence.category.isin(categories)]
    old_op[group] = {metric: float(rows[metric].mean()) for metric in ['card_hit_at_3', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']}
for system in systems:
    checks = {}
    for group, metric in guardrail_metrics:
        old_metric = 'card_hit_at_3' if group in {'evidence', 'numeric', 'semantic'} and metric == 'card_hit_at_3' else metric
        if metric in old_op[group]:
            checks[group + ':' + metric] = summary_value(system, group, metric) + 1e-12 >= old_op[group][metric]
    candidate_checks.setdefault(system, {})['vs_old_operational_prototype'] = checks
    candidate_checks[system]['pass_vs_old_operational_prototype'] = all(checks.values())
all_eligible = [system for system in ['all_gte', 'all_bge'] if candidate_checks[system]['pass_vs_new_no_reranker']]
pool = all_eligible if all_eligible else [system for system in ['selective_gte', 'selective_bge'] if candidate_checks[system]['pass_vs_new_no_reranker']]
selected = max(pool, key=lambda system: tuple(summary_value(system, group, metric) for group, metric in [('evidence', 'hit_at_3'), ('evidence', 'recall_at_5'), ('evidence', 'mrr_at_5'), ('evidence', 'ndcg_at_5'), ('card', 'card_hit_at_3'), ('card', 'card_mrr_at_5')])) if pool else 'no_reranker'
decision = {'status': 'development_only_not_eligible_for_promotion', 'selection_path': 'all_query' if selected.startswith('all_') else 'selective' if selected.startswith('selective_') else 'retain_no_reranker', 'selected_development_candidate': selected, 'candidate_guardrails': candidate_checks, 'old_operational_prototype': old_op, 'cross_corpus_recall_ndcg': 'diagnostic_only', 'resource_measurement_status': 'shared_gpu_measurement_non_confirmatory'}
(OUT / 'decision.json').write_text(json.dumps(decision, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary_payload = {'schema_version': 'structural_reranker_comparison_v1', 'rows': summary.replace({np.nan: None}).to_dict('records'), 'decision': decision, 'metric_korean': {'Hit@3': '상위 3개 안에 관련 결과 존재', 'Recall@5': '전체 관련 결과 중 상위 5개 회수 비율', 'MRR@5': '첫 관련 결과 순위의 역수', 'nDCG@5': '관련 결과가 앞쪽에 배치된 정도', 'Card Hit@3': '상위 3개 안에 기대 카드 존재', 'Card MRR@5': '기대 카드 첫 순위의 역수'}}
(OUT / 'summary.json').write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

source_final = hash_files(source_paths)
gte_final, bge_final, custom_final, custom_hub_final = tree_state(GTE), tree_state(BGE), tree_state(CUSTOM), tree_state(CUSTOM_HUB)
assert source_final == source_before
assert gte_final == gte_before and bge_final == bge_before and custom_final == custom_before and custom_hub_final == custom_hub_before
output_paths = sorted(path for path in OUT.iterdir() if path.is_file() and path.name not in {'integrity.json'})
output_hashes = {path.name: sha256_file(path) for path in output_paths}
integrity = {'status': 'passed', 'source_hashes_before': source_before, 'source_hashes_after': source_final, 'source_unchanged': True, 'model_cache_unchanged': True, 'query_count': 30, 'chunk_count': 147, 'pair_score_rows': 3000, 'pair_scores_finite': True, 'per_query_rows': 300, 'summary_rows': 50, 'candidate_ceiling_rows': 60, 'top20_exact_prefix': True, 'permutation_exact': True, 'baseline_22_exact': True, 'classification_21_exact': True, 'no_leak_allowlist': True, 'gpu_model_scoring_executed': True, 'network_api_download_new_embedding_chroma': 0, 'output_hashes_excluding_self': output_hashes}
(OUT / 'integrity.json').write_text(json.dumps(integrity, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
readme = f'''# 23 구조 청킹 GTE/BGE reranker 비교

개발 질의 30개에서 새 구조 청킹의 고정 RRF Top20/Top50 후보를 GTE와 BGE로 다시 정렬한 단일 실행이다. proper/numeric은 RRF를 유지하고 semantic만 reranker를 쓰는 selective 경로도 함께 비교했다.

두 모델 입력은 issuer, card_name, heading_path, body만으로 만든 임시 제목 보강 문자열이다. gold/relevance 필드는 입력에 쓰지 않았다. raw logit만 정렬에 사용했고 RRF 점수와 섞지 않았다.

Hit@3는 상위 3개 안에 관련 결과가 하나라도 있는지, Recall@5는 전체 관련 결과 중 상위 5개가 회수한 비율, MRR@5는 첫 관련 결과가 얼마나 앞에 있는지, nDCG@5는 관련 결과가 앞쪽에 모였는지를 뜻한다. Card Hit/MRR은 같은 계산을 기대 카드 기준으로 한다.

선택 결과: {selected}. 이는 개발셋 비교일 뿐 운영 확정이나 holdout 통과가 아니다. 이전 청킹과의 Recall/nDCG 비교는 corpus와 관련 문서 분모가 달라 진단용이다. GPU0에서 ollama가 함께 실행된 상태였으므로 load/scoring 시간, 처리량, VRAM은 shared_gpu_measurement_non_confirmatory이다. network/API/download/new embedding/Chroma query 비용은 0이다.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8')
integrity['output_hashes_excluding_self'] = {path.name: sha256_file(path) for path in output_paths}
(OUT / 'integrity.json').write_text(json.dumps(integrity, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'selected': selected, 'summary_rows': len(summary), 'per_query_rows': len(per_query), 'pair_score_rows': len(pair_scores)}, ensure_ascii=False, indent=2))


{
  "selected": "all_bge",
  "summary_rows": 50,
  "per_query_rows": 300,
  "pair_score_rows": 3000
}


## Follow-up — 저장 점수 기반 Top10 후보 제한

이 후속은 결과를 보기 전에 Top10을 원 RRF Top50의 앞 10개 후보로 고정한다. 저장된 GTE/BGE raw logit만 사용하며 Top50 rerank 결과를 잘라 쓰지 않는다. GPU·모델·외부 호출 없이 새 셀만 fresh kernel에서 실행한다. 비용은 모델별 30×10=300개 후보 조회라는 구조적 수만 기록하며 latency/VRAM 절감률은 측정하거나 주장하지 않는다.


In [1]:
from pathlib import Path
import hashlib, json, math, os
import numpy as np
import pandas as pd

cwd = Path.cwd()
ROOT = cwd if cwd.name == 'PickCardU' else cwd.parent if cwd.name == 'notebooks' and cwd.parent.name == 'PickCardU' else (_ for _ in ()).throw(RuntimeError('cwd must be repo root or its notebooks directory'))
OUT = ROOT / 'notebooks/data/23_structural_chunking_reranker_comparison'
NB_PATH = ROOT / 'notebooks/23_structural_chunking_reranker_comparison.ipynb'
S13 = ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
S22 = ROOT / 'notebooks/data/22_structural_heading_chunking_ablation'
CONFIG = 'k1_1.5_b_0.75_vector_0.4_bm25_0.6'
BASE_CELL_COUNT = 7
PRE_FOLLOWUP_NOTEBOOK_RAW_SHA256 = 'c222c24daa079355ed5eec507529469b172385aab88d7c0cd8f3f0b59053df79'
BASE_CELLS_SHA256 = '698ad965eea93f7dac88e60db19d88781feeb45e8daa61855846f1b0e46e29ee'
FROZEN_OUTPUT_HASHES = {
    'augmented_input_audit.csv': '74f91d66b95d3e6928318c9a7ac1654c7d2ac1f421edb486808a41e94dd65c63',
    'candidate_ceiling.csv': 'b678c38237b0dd78dc4d897b2d70fab19a3f648f81a4b07f7a9e9a32d199e687',
    'cross_pipeline_comparison.csv': 'd5983d51e737c68cb0bab3a933a5d67831214951e5e210abec4ca686853f2761',
    'cross_pipeline_summary.csv': 'b96c80dc3eea15b45a0b5d7b0b47314cd6c3d5a9bf2a8c600351e61531326355',
    'decision.json': '858b837a7b0af1c31488f30d75e1e47d5829d42edf06edb5253cb6300475028e',
    'evaluation_contract.json': '9dca3a0bef8cb46b09c0124a7d8f376b610fa0b1afc2900ed5bafd7642d4d14d',
    'integrity.json': 'fa24911fd98d9e317908eb8e9f48437c2892dba1bf3b6df478239a059f143359',
    'pair_scores.csv': 'f3b0d0c02e37a66e6902374eb046d31c8270906efcfdd86e50a7e775c8093083',
    'paired_deltas.csv': '6812af757b9ae06a08472ab81d4ad7f5b01ac99c65b691aabe6e4359a25e6941',
    'paired_wlt.csv': '003b4aaf6d71a2331f03b8335cf9e59c99bf43b7965e041aa67ccd775e835a10',
    'per_query_metrics.csv': '56a6ff5a633a7b6aa2e333f79d3d6903a66d2e9b0118836810410ccc9b39431a',
    'preflight.json': 'f3a2b0fefe820d0415bbdc2ca6e6b280d1eab30291093cbc6e62740881f5d2ba',
    'preflight_classification.csv': 'aa6025672ec4ebd47d8204e283606769ab2337f49151b8d54727b967037d7bb1',
    'resources.json': '9f66131807ec599e54e9b067db7d43a8e439c7d841064fe2347b6e3c2521a461',
    'summary.csv': 'a01424b156c9ba486d661978e7b2781971bba5e2418896389f47c56ce8bfa311',
    'summary.json': 'fb0563c87302afd9653788ab211ad42a765433c43c01b6d8b5447e1600e734a6',
}

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def frozen_outputs_state():
    return {name: sha256_file(OUT / name) for name in FROZEN_OUTPUT_HASHES}

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

notebook = json.loads(NB_PATH.read_text(encoding='utf-8'))
base_cells_bytes = json.dumps(notebook['cells'][:BASE_CELL_COUNT], ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()
assert hashlib.sha256(base_cells_bytes).hexdigest() == BASE_CELLS_SHA256
assert frozen_outputs_state() == FROZEN_OUTPUT_HASHES
source_paths = [S13 / 'retrieval_per_query.csv', S22 / 'chunks.jsonl', S22 / 'relevance_sets.jsonl', S22 / 'followup2_rankings.csv']
source_before = {str(path.relative_to(ROOT)): sha256_file(path) for path in source_paths}

contract = {
    'schema_version': 'structural_reranker_top10_followup_v1',
    'provenance': 'Codex coder agent; post-hoc offline Top10 follow-up',
    'pre_followup_notebook_raw_sha256': PRE_FOLLOWUP_NOTEBOOK_RAW_SHA256,
    'pre_followup_base_cells_sha256': BASE_CELLS_SHA256,
    'candidate_definition': 'first 10 chunks of original RRF Top50, then rerank only within that set',
    'forbidden_shortcut': 'do not truncate the reranked Top50 to 10',
    'ranking': 'saved raw logit descending; tie original RRF rank then chunk_id; no score mixing',
    'systems': ['no_reranker', 'all_gte', 'all_bge', 'selective_gte', 'selective_bge'],
    'primary': ['no_reranker', 'all_bge'],
    'groups': ['all', 'card', 'evidence', 'numeric', 'semantic'],
    'metrics': ['card_hit_at_3', 'card_mrr_at_5', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5'],
    'guardrail_priority': ['hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5'],
    'guardrails': 'all-BGE Top10 nonregression vs all-BGE Top20, all-BGE Top50, and old operational prototype across evidence/numeric/semantic relevance metrics and card metrics; numeric nonregression required',
    'execution': {'environment': 'skn25', 'fresh_kernel_new_cell_only': True, 'gpu': 0, 'model_loads': 0, 'scoring': 0, 'network_api_download_new_embedding_chroma_package_install': 0},
    'claim': 'development follow-up only; not eligible for promotion',
}
(OUT / 'top10_followup_contract.json').write_text(json.dumps(contract, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

queries = pd.read_csv(S13 / 'retrieval_per_query.csv')
queries = queries[queries.method == 'keyword'].copy()
assert len(queries) == queries.query_id.nunique() == 30
query_map = queries.set_index('query_id').to_dict('index')
chunks = load_jsonl(S22 / 'chunks.jsonl')
chunks_by_id = {row['chunk_id']: row for row in chunks}
assert len(chunks_by_id) == 147
relevance_rows = [row for row in load_jsonl(S22 / 'relevance_sets.jsonl') if row['configuration'] == 'structural_heading_path']
assert len(relevance_rows) == 20
relevance = {row['query_id']: set(row['relevant_chunk_ids']) for row in relevance_rows}
classifications = pd.read_csv(OUT / 'preflight_classification.csv')
assert len(classifications) == 30 and classifications.correct.all()
class_map = classifications.set_index('query_id').predicted_category.to_dict()
rankings = pd.read_csv(S22 / 'followup2_rankings.csv')
rankings = rankings[rankings.configuration == CONFIG].copy()
assert len(rankings) == 30
rankings['fused_top50_chunk_ids'] = rankings.fused_top50_chunk_ids.map(json.loads)
assert all(len(ids) == len(set(ids)) == 50 for ids in rankings.fused_top50_chunk_ids)
top10_by_query = {row.query_id: row.fused_top50_chunk_ids[:10] for row in rankings.itertuples(index=False)}
scores = pd.read_csv(OUT / 'pair_scores.csv')
assert len(scores) == 3000 and np.isfinite(scores.raw_logit).all()
top10_scores = scores[scores.original_rrf_rank <= 10].copy()
assert len(top10_scores) == 600 and not top10_scores.duplicated(['model', 'query_id', 'chunk_id']).any()
for (model, query_id), frame in top10_scores.groupby(['model', 'query_id']):
    assert frame.original_rrf_rank.tolist() == list(range(1, 11))
    assert frame.chunk_id.tolist() == top10_by_query[query_id]
score_map = {(row.model, row.query_id, row.chunk_id): row.raw_logit for row in top10_scores.itertuples(index=False)}

def reciprocal_rank(ids, relevant):
    return next((1.0 / rank for rank, chunk_id in enumerate(ids[:5], 1) if chunk_id in relevant), 0.0)

def ndcg_at_5(ids, relevant):
    dcg = sum(1.0 / np.log2(rank + 1) for rank, chunk_id in enumerate(ids[:5], 1) if chunk_id in relevant)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return dcg / idcg if idcg else np.nan

systems = ['no_reranker', 'all_gte', 'all_bge', 'selective_gte', 'selective_bge']
rows = []
for query_id, base in top10_by_query.items():
    query = query_map[query_id]
    model_rankings = {model: sorted(base, key=lambda chunk_id: (-score_map[(model, query_id, chunk_id)], base.index(chunk_id) + 1, chunk_id)) for model in ['gte', 'bge']}
    assert all(len(ids) == 10 and set(ids) == set(base) for ids in model_rankings.values())
    system_rankings = {'no_reranker': base, 'all_gte': model_rankings['gte'], 'all_bge': model_rankings['bge'], 'selective_gte': model_rankings['gte'] if class_map[query_id] == 'semantic' else base, 'selective_bge': model_rankings['bge'] if class_map[query_id] == 'semantic' else base}
    card_relevant = {chunk_id for chunk_id, chunk in chunks_by_id.items() if chunk['metadata']['card_key'] == query['expected_card']}
    answer_relevant = relevance.get(query_id, set())
    for system, ranked in system_rankings.items():
        rows.append({'depth': 10, 'system': system, 'query_id': query_id, 'question_group': 'card' if query['category'] == 'proper_noun' else 'evidence', 'category': query['category'], 'candidate_count': 10, 'route': 'reranker' if system.startswith('all_') or system.startswith('selective_') and class_map[query_id] == 'semantic' else 'no_reranker', 'card_hit_at_3': int(bool(set(ranked[:3]) & card_relevant)), 'card_mrr_at_5': reciprocal_rank(ranked, card_relevant), 'hit_at_3': int(bool(set(ranked[:3]) & answer_relevant)) if answer_relevant else np.nan, 'recall_at_5': len(set(ranked[:5]) & answer_relevant) / len(answer_relevant) if answer_relevant else np.nan, 'mrr_at_5': reciprocal_rank(ranked, answer_relevant) if answer_relevant else np.nan, 'ndcg_at_5': ndcg_at_5(ranked, answer_relevant), 'relevant_count': len(answer_relevant) if answer_relevant else np.nan, 'top5_chunk_ids': json.dumps(ranked[:5], ensure_ascii=False), 'top10_chunk_ids': json.dumps(ranked, ensure_ascii=False), 'source_top10_rrf_chunk_ids': json.dumps(base, ensure_ascii=False)})
per_query10 = pd.DataFrame(rows)
assert len(per_query10) == 150 and per_query10.groupby('system').size().eq(30).all()
assert not per_query10.duplicated(['system', 'query_id']).any()
for metric in contract['metrics']:
    assert per_query10[metric].dropna().between(0, 1).all()
existing_per_query = pd.read_csv(OUT / 'per_query_metrics.csv')
for depth in (20, 50):
    old = existing_per_query[(existing_per_query.depth == depth) & (existing_per_query.system == 'no_reranker')].sort_values('query_id').reset_index(drop=True)
    new = per_query10[per_query10.system == 'no_reranker'].sort_values('query_id').reset_index(drop=True)
    assert new.top5_chunk_ids.map(json.loads).tolist() == old.top5_chunk_ids.map(json.loads).tolist()
    for metric in contract['metrics']:
        assert np.allclose(new[metric].to_numpy(float), old[metric].to_numpy(float), rtol=0, atol=1e-12, equal_nan=True)
per_query10.to_csv(OUT / 'top10_followup_per_query.csv', index=False)

ceiling_rows = []
for query_id, ids in top10_by_query.items():
    query = query_map[query_id]
    card_ids = {chunk_id for chunk_id, chunk in chunks_by_id.items() if chunk['metadata']['card_key'] == query['expected_card']}
    relevant = relevance.get(query_id, set())
    ceiling_rows.append({'query_id': query_id, 'category': query['category'], 'depth': 10, 'candidate_count': 10, 'expected_card_candidate_hit': int(bool(set(ids) & card_ids)), 'expected_card_candidate_count': len(set(ids) & card_ids), 'answer_candidate_hit': int(bool(set(ids) & relevant)) if relevant else np.nan, 'answer_candidate_recall': len(set(ids) & relevant) / len(relevant) if relevant else np.nan, 'missing_relevant_ids': json.dumps(sorted(relevant - set(ids)), ensure_ascii=False)})
ceiling10 = pd.DataFrame(ceiling_rows)
assert len(ceiling10) == 30 and ceiling10.candidate_count.eq(10).all()
ceiling10.to_csv(OUT / 'top10_followup_candidate_ceiling.csv', index=False)

group_masks = {'all': lambda frame: pd.Series(True, index=frame.index), 'card': lambda frame: frame.category == 'proper_noun', 'evidence': lambda frame: frame.category != 'proper_noun', 'numeric': lambda frame: frame.category == 'numeric_condition', 'semantic': lambda frame: frame.category == 'semantic'}
summary_rows = []
for system, frame in per_query10.groupby('system'):
    for group, mask in group_masks.items():
        selected = frame[mask(frame)]
        row = {'depth': 10, 'system': system, 'group': group, 'denominator': len(selected)}
        for metric in contract['metrics']:
            row[metric] = float(selected[metric].mean()) if selected[metric].notna().any() else np.nan
            row[metric + '_denominator'] = int(selected[metric].notna().sum())
        summary_rows.append(row)
summary10 = pd.DataFrame(summary_rows)
assert len(summary10) == 25
existing_summary = pd.read_csv(OUT / 'summary.csv')
assert len(existing_summary) == 50 and set(existing_summary.depth) == {20, 50}
combined_summary = pd.concat([summary10.assign(provenance='top10_offline_followup'), existing_summary.assign(provenance='original_23')], ignore_index=True).sort_values(['depth', 'system', 'group']).reset_index(drop=True)
assert len(combined_summary) == 75 and set(combined_summary.depth) == {10, 20, 50}
combined_summary.to_csv(OUT / 'top10_followup_summary.csv', index=False)

paired_rows = []
baseline = per_query10[per_query10.system == 'no_reranker'].set_index('query_id')
for system in systems[1:]:
    candidate = per_query10[per_query10.system == system].set_index('query_id')
    for query_id in baseline.index:
        for metric in contract['metrics']:
            left, right = baseline.at[query_id, metric], candidate.at[query_id, metric]
            if pd.notna(left) and pd.notna(right):
                paired_rows.append({'depth': 10, 'comparison': system + '_vs_no_reranker', 'system': system, 'query_id': query_id, 'category': baseline.at[query_id, 'category'], 'metric': metric, 'baseline': left, 'candidate': right, 'delta': right - left})
paired10 = pd.DataFrame(paired_rows)
paired10.to_csv(OUT / 'top10_followup_paired.csv', index=False)
wlt_rows = []
for (comparison, metric), frame in paired10.groupby(['comparison', 'metric']):
    for group, categories in {'all': None, 'card': {'proper_noun'}, 'evidence': {'numeric_condition', 'semantic'}, 'numeric': {'numeric_condition'}, 'semantic': {'semantic'}}.items():
        selected = frame if categories is None else frame[frame.category.isin(categories)]
        if len(selected):
            wlt_rows.append({'depth': 10, 'comparison': comparison, 'group': group, 'metric': metric, 'denominator': len(selected), 'wins': int((selected.delta > 1e-12).sum()), 'losses': int((selected.delta < -1e-12).sum()), 'ties': int((selected.delta.abs() <= 1e-12).sum()), 'mean_delta': float(selected.delta.mean())})
wlt10 = pd.DataFrame(wlt_rows)
assert (wlt10.wins + wlt10.losses + wlt10.ties == wlt10.denominator).all()
wlt10.to_csv(OUT / 'top10_followup_wlt.csv', index=False)

def value(frame, depth, system, group, metric):
    selected = frame[(frame.depth == depth) & (frame.system == system) & (frame.group == group)]
    assert len(selected) == 1
    return float(selected.iloc[0][metric])

guardrail_metrics = [(group, metric) for group in ['evidence', 'numeric', 'semantic'] for metric in ['hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5']] + [('card', 'card_hit_at_3'), ('card', 'card_mrr_at_5')]
top10_checks = {}
for reference_depth in [20, 50]:
    checks = {group + ':' + metric: value(combined_summary, 10, 'all_bge', group, metric) + 1e-12 >= value(combined_summary, reference_depth, 'all_bge', group, metric) for group, metric in guardrail_metrics}
    top10_checks['vs_top' + str(reference_depth)] = {'checks': checks, 'pass': all(checks.values())}
old_operational = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))['old_operational_prototype']
old_checks = {group + ':' + metric: value(combined_summary, 10, 'all_bge', group, metric) + 1e-12 >= old_operational[group][metric] for group, metric in guardrail_metrics}
top10_checks['vs_old_operational_prototype'] = {'checks': old_checks, 'pass': all(old_checks.values()), 'cross_corpus_recall_ndcg': 'diagnostic_only'}
all_pass = all(item['pass'] for item in top10_checks.values())
decision = {'status': 'top10_quality_guardrails_pass_dev_followup' if all_pass else 'top10_quality_guardrails_fail_dev_followup', 'all_bge_top10_passes_all_references': all_pass, 'guardrails': top10_checks, 'quality_priority': ['hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5'], 'numeric_nonregression_required': True, 'claim': ['development_followup_only', 'not_eligible_for_promotion', 'no_top10_latency_or_vram_reduction_claim']}
(OUT / 'top10_followup_decision.json').write_text(json.dumps(decision, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary_json = {'schema_version': 'structural_reranker_top10_followup_summary_v1', 'rows': combined_summary.replace({np.nan: None}).to_dict('records'), 'decision': decision}
(OUT / 'top10_followup_summary.json').write_text(json.dumps(summary_json, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
resources = {'candidate_pairs_per_model': 300, 'models_in_saved_score_cache': 2, 'total_saved_score_lookups': 600, 'gpu_model_load_scoring': 0, 'top10_only_latency_vram_measured': False, 'latency_or_vram_reduction_claim': False, 'network_api_download_new_embedding_chroma_package_install': 0}
(OUT / 'top10_followup_resources.json').write_text(json.dumps(resources, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

readme_path = OUT / 'README.md'
readme_base = readme_path.read_text(encoding='utf-8').split('\n## Top10 offline follow-up')[0].rstrip()
readme_followup = '''

## Top10 offline follow-up

원 RRF Top50의 앞 10개만 후보로 제한한 뒤, 저장된 raw logit으로 그 10개 안에서 다시 정렬했다. Top50 rerank 결과를 잘라 쓰지 않았다. GPU·모델·외부 호출은 없었고 모델별 후보 구조 수는 30×10=300개다. Top10 전용 latency/VRAM은 측정하지 않았으므로 시간·메모리 절감률을 주장하지 않는다. 판정은 개발셋 후속 진단이며 운영 승격 근거가 아니다.
'''
readme_path.write_text(readme_base + readme_followup, encoding='utf-8')

assert frozen_outputs_state() == FROZEN_OUTPUT_HASHES
source_after = {str(path.relative_to(ROOT)): sha256_file(path) for path in source_paths}
assert source_after == source_before
followup_names = ['top10_followup_contract.json', 'top10_followup_per_query.csv', 'top10_followup_summary.csv', 'top10_followup_summary.json', 'top10_followup_paired.csv', 'top10_followup_wlt.csv', 'top10_followup_candidate_ceiling.csv', 'top10_followup_decision.json', 'top10_followup_resources.json']
followup_hashes = {name: sha256_file(OUT / name) for name in followup_names}
integrity = {'status': 'passed', 'base_cells_sha256': BASE_CELLS_SHA256, 'base_cells_preserved': True, 'pre_followup_notebook_raw_sha256_recorded': PRE_FOLLOWUP_NOTEBOOK_RAW_SHA256, 'frozen_output_hashes_before_after': FROZEN_OUTPUT_HASHES, 'frozen_outputs_preserved': True, 'source_hashes_before': source_before, 'source_hashes_after': source_after, 'source_unchanged': True, 'top10_from_original_rrf_prefix': True, 'top50_rerank_truncation_used': False, 'saved_score_source_exact': True, 'pair_score_rows_used': 600, 'per_query_rows': 150, 'combined_summary_rows': 75, 'paired_rows': len(paired10), 'wlt_rows': len(wlt10), 'candidate_ceiling_rows': 30, 'ranking_permutation_exact': True, 'metric_ranges_pass': True, 'gpu_model_network_external_calls': 0, 'followup_output_hashes_excluding_self': followup_hashes, 'readme_sha256': sha256_file(readme_path)}
(OUT / 'top10_followup_integrity.json').write_text(json.dumps(integrity, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'decision': decision['status'], 'per_query_rows': len(per_query10), 'combined_summary_rows': len(combined_summary), 'paired_rows': len(paired10), 'wlt_rows': len(wlt10)}, ensure_ascii=False, indent=2))


{
  "decision": "top10_quality_guardrails_pass_dev_followup",
  "per_query_rows": 150,
  "combined_summary_rows": 75,
  "paired_rows": 560,
  "wlt_rows": 104
}
